<a href="https://colab.research.google.com/github/rudrajoshi07/inference-lab/blob/main/01_baseline_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi

Thu Oct  1 06:01:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/inference-lab/results

Mounted at /content/drive


In [5]:
!pip uninstall -y torchaudio
!pip install -q vllm openai

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 85.3 MB/

In [6]:
import subprocess, time, requests

server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "Qwen/Qwen2.5-1.5B-Instruct",
     "--dtype", "half",
     "--max-model-len", "2048",
     "--gpu-memory-utilization", "0.85",
     "--enforce-eager",
     "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT)

for i in range(60):
    if server.poll() is not None:
        print("server exited early")
        break
    try:
        if requests.get("http://localhost:8000/health").status_code == 200:
            print("ready")
            break
    except Exception:
        pass
    time.sleep(5)

ready


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="x")

r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "Explain keyboard in one sentance."}],
    max_tokens=60, temperature=0)
print(r.choices[0].message.content)

A keyboard is an input device used to enter text and commands into a computer or other electronic devices.


In [8]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 12945 MiB, 15360 MiB


In [11]:
!mkdir -p serve
!touch serve/baseline.md

In [12]:
%%writefile serve/baseline.md
# Baseline Server Setup
This document tr- Hardware: Colab free T4 (15 GB)
- Model: Qwen/Qwen2.5-1.5B-Instruct, FP16
- Command:
  python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 2048 --gpu-memory-utilization 0.85 --enforce-eager --port 8000
- GPU memory used (nvidia-smi): 12945 MiB
- Test request: worked

Overwriting serve/baseline.md


In [23]:
!git config --global user.name "rudrajoshi07"
!git config --global user.email "rudrajoshi7905@gmail.com"